# 02 · Segmentación pulmonar

Este notebook documenta el entrenamiento y la evaluación de los métodos de segmentación
pulmonar del proyecto.

El objetivo **no es obtener el mejor segmentador**, sino **medir si la técnica de
segmentación cambia las características radiómicas y el desempeño de los clasificadores
de TB**. La segmentación es un paso intermedio: todas las técnicas que pasen el control
de calidad continúan al notebook de extracción de características.

De eso derivan dos exigencias de diseño:
1. Todas las técnicas se entrenan y evalúan **en igualdad de condiciones**: mismos datos,
   misma partición, misma función de pérdida, mismo optimizador. Las diferencias
   posteriores se atribuyen a la técnica y no al protocolo.
2. Agregar un método nuevo no debe obligar a rehacer las etapas siguientes.

**Estado de este notebook:** U-Net y Attention U-Net entrenadas y comparadas.
MedSAM y un método de referencia clásico (k-means u otro) se añadirán en la siguiente
iteración.

**Relación con el resto del proyecto**

```
01b (Montgomery-Shenzhen) ──► 02 (este notebook) ──► 03 (radiómica) ──► ...
                                        │
                               entrena U-Net y Att-UNet
                               genera máscaras para TBX11K
```

| Sección | Contenido |
|---------|-----------|
| 1 | Configuración común a todas las corridas |
| 2 | Datos de entrenamiento |
| 3 | Arquitecturas |
| 4 | Protocolo de entrenamiento |
| 5 | Curvas de aprendizaje |
| 6 | Calidad en el conjunto de prueba |
| 7 | Siguientes pasos |

## 1 · Configuración común

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from tbseg.config import IMAGE_SIZE, RESULTS_DIR, SEGTRAIN_CSV, SEGTRAIN_DIR
from tbseg.segmentation import train as T

SEG_RESULTS_DIR = RESULTS_DIR / "segmentation"

# Entreno cada arquitectura con tres semillas distintas para separar la variación
# debida al azar del entrenamiento de la variación debida a la arquitectura.
SEEDS = [42, 43, 44]

# ── Tabla de parámetros ────────────────────────────────────────────────────────
# Los importo directamente del módulo de entrenamiento para garantizar que lo que
# se muestra aquí coincide con lo que se usó al entrenar.
config = pd.Series({
    "Resolución":          f"{IMAGE_SIZE}×{IMAGE_SIZE}",
    "Tamaño de lote":      T.BATCH_SIZE,
    "Optimizador":         f"Adam (lr={T.LEARNING_RATE})",
    "Pérdida":             "BCE + Dice",
    "Épocas máximas":      T.MAX_EPOCHS,
    "Detención temprana":  f"{T.PATIENCE} épocas sin mejorar el Dice de validación",
    "Canales por nivel":   str(T.CHANNELS),
    "Umbral de binarización": T.THRESHOLD,
    "Semillas":            ", ".join(map(str, SEEDS)),
}).to_frame("valor")
config.index.name = "parámetro"
display(config)

# ── Estilo de figuras ──────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
C_TRUE, C_PRED  = "#2a78d6", "#eb6834"   # azul: máscara real | naranja: predicción
ARCH_NAMES  = {"unet": "U-Net", "attention_unet": "Attention U-Net"}
ARCH_COLORS = {"unet": "#2a78d6", "attention_unet": "#1baf7a"}

### Aumento de datos

El aumento se aplica solo al conjunto de entrenamiento y está definido en
`train.augment`. Las transformaciones que uso son:

| Transformación | Rango | Justificación |
|----------------|-------|---------------|
| Rotación | ±10° | Inclinación del paciente frente al detector |
| Escala | 0,9–1,1 | Distancia y contextura del paciente |
| Traslación | hasta 5 % | Encuadre del técnico |
| Brillo | ±20 % | Exposición del equipo |
| Contraste | ±0,1 | Exposición del equipo |

Las transformaciones geométricas se aplican **igual a la imagen y a su máscara**;
las de intensidad solo a la imagen.

Los rangos son pequeños a propósito: una radiografía de tórax es una proyección
estandarizada y deformarla más generaría imágenes que no existen en la práctica.
**No uso volteo horizontal** porque pondría el corazón a la derecha.

## 2 · Datos de entrenamiento

TBX11K no tiene máscaras de pulmón, así que entreno con **Montgomery + Shenzhen**:
704 radiografías con máscaras trazadas manualmente (exploradas en el notebook 01b).

El script `python -m tbseg.segmentation.prepare` ya las dejó a 512 × 512 px y
escribió la partición en `data/metadata/segtrain_split.csv`.

La partición es:
- **train (70 %)**: lo que la red usa para aprender.
- **val (15 %)**: vigila el entrenamiento y selecciona la mejor época.
- **test (15 %)**: se usa **una sola vez**, al final, para reportar el desempeño.

La partición se hace **por paciente**: las imágenes de los 2 pacientes de Montgomery
con varias visitas caen todas en la misma partición para evitar fuga de información.
Está estratificada por fuente (Montgomery/Shenzhen) y diagnóstico (sano/TB),
y es **la misma para todas las corridas y arquitecturas**.

In [ ]:
split = pd.read_csv(SEGTRAIN_CSV)
print(f"{len(split)} imágenes de {split['patient_id'].nunique()} pacientes")
display(
    pd.crosstab(
        [split["source"], split["tb"].map({0: "sano", 1: "TB"})],
        split["split"],
        margins=True, margins_name="total"
    )
)

In [ ]:
# Visualizo tres ejemplos del conjunto de entrenamiento con su máscara manual,
# para confirmar que la carga y el redimensionado funcionan correctamente.
ids = split[split["split"] == "train"].sample(3, random_state=0)["image_id"]

fig, axes = plt.subplots(1, 3, figsize=(10, 3.6))
for ax, image_id in zip(axes, ids):
    img, mask = T.load_pair(image_id)
    ax.imshow(img, cmap="gray")
    ax.contour(mask, levels=[0.5], colors=C_TRUE, linewidths=1.5)
    ax.set_title(image_id, fontsize=9)
    ax.axis("off")
fig.suptitle("Ejemplos de entrenamiento (contorno azul = máscara manual)", x=0.01, ha="left")
fig.tight_layout()
plt.show()

## 3 · Arquitecturas

In [ ]:
# Uso las implementaciones de MONAI, una librería especializada en imagen médica.
# Paso los mismos argumentos a todas las arquitecturas para que la comparación
# sea justa: la única diferencia es la estructura interna de la red.

display(pd.DataFrame([
    {
        "arquitectura": arch,
        "parámetros": f"{sum(p.numel() for p in T.build_model(arch).parameters()):,}"
    }
    for arch in T.ARCHITECTURES
]))

Las dos arquitecturas comparten el mismo esquema codificador-decodificador:

- **U-Net** (Ronneberger et al., 2015): el estándar en segmentación médica.
  Un codificador que reduce la resolución ganando contexto, un decodificador
  que la recupera, y **conexiones de salto** que devuelven el detalle espacial perdido.
  Funciona bien con pocos datos, que es exactamente la situación de este proyecto.

- **Attention U-Net** (Oktay et al., 2018): la misma estructura, pero agrega
  **compuertas de atención** en las conexiones de salto. Antes de combinar la
  información del codificador con la del decodificador, la compuerta pondera
  cada región según su relevancia. La idea es que la red atenúe el fondo
  (hombros, abdomen) y se concentre en el pulmón.

Una explicación detallada de U-Net, pieza por pieza, está en
[`tutorial_unet.ipynb`](tutorial_unet.ipynb).

## 4 · Protocolo de entrenamiento

El entrenamiento **no se ejecuta dentro del notebook**: son corridas largas que
deben poder lanzarse, interrumpirse y repetirse sin depender del estado del kernel.
La rutina completa vive en `src/tbseg/segmentation/train.py`.

### Qué hace cada corrida

1. **Carga los datos** (`build_loaders`): las 704 imágenes a 512 × 512, repartidas
   según `segtrain_split.csv`. Solo el conjunto de entrenamiento baraja el orden
   y aplica aumentos.
2. **Crea la red** (`build_model`) y la mueve a la GPU si hay una disponible.
3. **Repite por época**:
   - Recorre el entrenamiento ajustando pesos (paso hacia adelante + gradiente).
   - Evalúa en validación sin ajustar pesos.
   - **Función de pérdida**: BCE + Dice. La entropía cruzada binaria (BCE) penaliza
     el acierto píxel a píxel; el Dice penaliza la superposición global entre
     la máscara predicha y la real. Usarlas juntas balancea ambos objetivos.
   - **Optimizador**: Adam con tasa de aprendizaje 0,001.
   - Si el Dice de validación mejora, guarda una copia de los pesos.
4. **Detención temprana**: para tras 10 épocas sin mejorar el Dice de validación,
   con un máximo de 60 épocas.
5. **Evalúa en el conjunto de prueba una sola vez**, con los pesos de la mejor época.

### Comandos para reproducir las corridas

Prueba rápida antes de una corrida larga (2 épocas, 24 imágenes, no guarda nada):

```bash
python -m tbseg.segmentation.train --arch unet --smoke
```

Corridas completas (las que analizo en este notebook):

```bash
for seed in 42 43 44; do
    python -m tbseg.segmentation.train --arch unet --seed $seed
done
for seed in 42 43 44; do
    python -m tbseg.segmentation.train --arch attention_unet --seed $seed
done
```

**Por qué tres semillas.** El entrenamiento tiene componentes aleatorios:
inicialización de pesos, orden de los lotes y aumentos. Con una sola corrida
no se puede saber si una diferencia entre arquitecturas es real o producto
del azar. Tres semillas permiten reportar media ± desviación y comparar esa
variación con la diferencia entre arquitecturas.

### Archivos que genera cada corrida

| Archivo | Contenido |
|---------|-----------|
| `models/<arch>_seed<n>.pt` | Pesos de la mejor época (no versionado) |
| `results/segmentation/history.csv` | Pérdida y Dice por época, todas las corridas |
| `results/segmentation/test_per_image.csv` | Dice e IoU por imagen de prueba |
| `results/segmentation/test_metrics.csv` | Una fila resumen por corrida |

## 5 · Curvas de aprendizaje

In [ ]:
# Leo el historial de todas las corridas. Cada fila tiene la época, la arquitectura,
# la semilla y las métricas de entrenamiento y validación de ese momento.
histories = pd.read_csv(SEG_RESULTS_DIR / "history.csv")
archs     = [a for a in ARCH_NAMES if a in set(histories["arch"])]

fig, axes = plt.subplots(len(archs), 2,
                          figsize=(12, 3.6 * len(archs)), squeeze=False)
for row, arch in zip(axes, archs):
    for ax, metric, title in [
        (row[0], "loss", "Pérdida (BCE + Dice)"),
        (row[1], "dice", "Dice"),
    ]:
        for seed, run in histories[histories["arch"] == arch].groupby("seed"):
            # Dibujo una línea por semilla; la diferencia entre ellas muestra
            # cuánto varía el entrenamiento solo por el componente aleatorio.
            ax.plot(run["epoch"], run[f"train_{metric}"],
                    color=C_TRUE, lw=1.5, alpha=0.7)
            ax.plot(run["epoch"], run[f"val_{metric}"],
                    color=C_PRED, lw=1.5, alpha=0.7)
        ax.set_title(f"{ARCH_NAMES[arch]} — {title}", loc="left")
        ax.set_xlabel("Época")
        ax.grid(alpha=0.3)
    row[0].set_ylabel("Pérdida")
    row[1].set_ylabel("Dice")
    row[1].set_ylim(0.8, 1.0)

# Leyenda compartida para el primer panel
axes[0, 0].plot([], [], color=C_TRUE, label="entrenamiento")
axes[0, 0].plot([], [], color=C_PRED, label="validación")
axes[0, 0].legend(frameon=False)
fig.suptitle("Una línea por semilla", x=0.01, ha="left")
fig.tight_layout()
plt.show()

# Tabla resumen: número de épocas, mejor Dice de validación y tiempo total
display(
    histories.groupby(["arch", "seed"]).agg(
        épocas        = ("epoch",     "max"),
        mejor_val_dice = ("val_dice",  "max"),
        minutos        = ("minutes",   "max"),
    ).round(3)
)

## 6 · Calidad de la segmentación en el conjunto de prueba

El conjunto de prueba no participó en ninguna decisión durante el entrenamiento:
ni para ajustar pesos, ni para la detención temprana, ni para seleccionar la mejor época.
Por eso su Dice es una estimación honesta del desempeño sobre radiografías nuevas
del mismo tipo (Montgomery y Shenzhen).

Uso dos métricas de superposición entre la máscara predicha y la manual:
- **Dice** = 2 · |intersección| / (|predicha| + |real|)
- **IoU** = |intersección| / |unión| — equivalente al Dice pero más estricta.

In [ ]:
metrics = pd.read_csv(SEG_RESULTS_DIR / "test_metrics.csv")
display(metrics)

# Resumen agregado por arquitectura sobre las tres semillas
summary = metrics.groupby("arch").agg(
    corridas      = ("seed",         "count"),
    dice_medio    = ("test_dice",     "mean"),
    dice_desv     = ("test_dice",     "std"),
    iou_medio     = ("test_iou",      "mean"),
    peor_imagen   = ("test_dice_min", "min"),
).round(4)
display(summary)

La desviación entre semillas cuantifica cuánto varía el resultado solo por el azar
del entrenamiento. Es la referencia para interpretar la diferencia entre arquitecturas:
si esa diferencia es menor que la desviación entre semillas, no es concluyente.

In [ ]:
# Distribución del Dice por imagen de prueba, cruzada con fuente y diagnóstico
per_image = pd.read_csv(SEG_RESULTS_DIR / "test_per_image.csv").merge(
    split[["image_id", "source", "tb"]], on="image_id"
)

display(
    per_image.groupby(["arch", "source"])["dice"]
             .describe()[["count", "mean", "std", "min"]]
             .round(3)
)

fig, ax = plt.subplots(figsize=(7, 3.4))
bins = np.linspace(per_image["dice"].min(), 1, 30)
for arch in [a for a in ARCH_NAMES if a in set(per_image["arch"])]:
    group = per_image[per_image["arch"] == arch]
    ax.hist(group["dice"], bins=bins, histtype="step",
            lw=2, color=ARCH_COLORS[arch], label=ARCH_NAMES[arch])
ax.set_title("Dice por imagen (las tres semillas juntas)", loc="left")
ax.set_xlabel("Dice")
ax.set_ylabel("Imágenes")
ax.legend(frameon=False, loc="upper left")
fig.tight_layout()
plt.show()

print("Peores casos:")
display(
    per_image.sort_values("dice")
             .head(10)[["arch", "image_id", "source", "tb", "seed", "dice", "iou"]]
             .round(3)
)

Los peores casos son los más informativos: los mismos identificadores aparecen
con las tres semillas, así que no son fallos aleatorios del entrenamiento, sino
radiografías inherentemente difíciles o con máscaras manuales discutibles.

### 6.1 Comparación entre arquitecturas

In [ ]:
# La pregunta no es cuál segmenta mejor, sino si las máscaras de cada técnica
# son suficientemente distintas como para cambiar las características radiómicas
# en la siguiente etapa.
#
# Comparo el Dice medio de ambas redes y lo pongo en perspectiva con la variación
# entre semillas: si la diferencia entre arquitecturas es mayor que esa variación,
# se debe a la arquitectura y no al azar del entrenamiento.

if not {"unet", "attention_unet"} <= set(per_image["arch"]):
    print("Falta entrenar Attention U-Net: la comparación se completa al tener las dos arquitecturas.")
else:
    dice     = metrics.groupby("arch")["test_dice"].agg(["mean", "std"])
    by_image = per_image.pivot_table(
        index="image_id", columns="arch", values="dice", aggfunc="mean"
    )
    diff = dice.loc["attention_unet", "mean"] - dice.loc["unet", "mean"]

    display(pd.Series({
        "Dice medio U-Net":                    f"{dice.loc['unet', 'mean']:.4f}",
        "Dice medio Attention U-Net":          f"{dice.loc['attention_unet', 'mean']:.4f}",
        "Diferencia (Attention − U-Net)":      f"{diff:+.4f}",
        "Variación entre semillas (desviación)": f"{dice['std'].max():.4f}",
        "Imágenes donde gana Attention U-Net": f"{(by_image['attention_unet'] > by_image['unet']).sum()} de {len(by_image)}",
    }).to_frame("valor"))

### Conclusiones de la comparación

- **Las máscaras difieren poco pero de forma real**: Attention U-Net tiene un Dice medio
  0,0015 mayor, tres veces la variación entre semillas (0,0005), y gana en 69 de las
  106 imágenes de prueba.
- **Los casos difíciles son los mismos para ambas** (`CHNCXR_0423_1`, `CHNCXR_0527_1`,
  `MCUCXR_0150_1`, todos TB). Ambas técnicas fallan de forma parecida en esos casos.
- **Costo y estabilidad**: Attention U-Net tiene el triple de parámetros y tarda unas
  cinco veces más por época. Su validación es más inestable al inicio (visible en las
  curvas de la sección 5), coherente con su normalización por lote.

**Para el proyecto:** ambas técnicas producen máscaras muy parecidas. La pregunta
que responde la siguiente etapa es si una diferencia de este tamaño alcanza a cambiar
las características radiómicas y el desempeño de los clasificadores, o si esas métricas
se mantienen estables entre métodos.

## 7 · Siguientes pasos

In [ ]:
# Esta etapa cierra con U-Net y Attention U-Net entrenadas y evaluadas.
# Las tres tareas pendientes antes de pasar al notebook de radiómica son:

# 1. Generar las máscaras de las 704 imágenes de Montgomery-Shenzhen
#    (y eventualmente las de TBX11K) con los modelos entrenados:
#       python -m tbseg.segmentation.predict --arch unet
#       python -m tbseg.segmentation.predict --arch attention_unet
#
# 2. Integrar MedSAM como tercer método de segmentación.
#    MedSAM no necesita entrenamiento, pero sí requiere cajas-prompt.
#    Las genero a partir de la posición media del pulmón descrita en 01a.
#
# 3. Evaluar un método clásico (k-means u otro no basado en redes neuronales)
#    como referencia de bajo costo computacional. Si produce máscaras de calidad
#    razonable, refuerza el argumento de que la segmentación es robusta.